In [36]:
!pip install pandas numpy matplotlib scikit-learn xgboost

In [37]:
import pandas as pd             #표 형태 데이터를 다루는 라이브러리 
import numpy as np              #수치 계산용 라이브러리 
import matplotlib.pyplot as plt # 그레프 시각화 라이브러리

# train_test_split: 데이터를 학습용과 검증용으로 나누는 함수
from sklearn.model_selection import train_test_split
# 회귀 모델 평가에 사용할 지표들 (MAE, MSE → RMSE, R2)
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
# ColumnTransformer: 컬럼 종류별로 다른 전처리를 적용하는 도구
from sklearn.compose import ColumnTransformer
# pipeline: 전처리와 모델 학습을 하나의 흐름으로 묶는 도구
from sklearn.pipeline import Pipeline
# OneHotEncoder: 문자형(범주형) 값을 숫자 형태로 바꾸는 도구
from sklearn.preprocessing import OneHotEncoder
# Simple Imputer: 비어있는 값 (결측치)를 정해진 규칙으로 채우는 도구
from sklearn.impute import SimpleImputer

from sklearn.model_selection import train_test_split
# XGBRegressor: 숫자 값을 예측하는 회귀용 XGBoost 모델
from xgboost import XGBRegressor

In [38]:
#그래프의 모양을 설정합니다.
plt.style.use('ggplot')

In [39]:
import matplotlib.font_manager as fm
import warnings

#  window 한글 폰트 설정

# matplotlib의 기본 글꼴은 DejaVu Sans인데, 이 글꼴은 한글은 재대로 표시하지 못합니다.
# 그래서 Windows 에 기본으로 설치된 '맑은 고딕' 을 그래프 글꼴로 지정합니다.
plt.rcParams['font.family'] = 'Malgun Gothic'

# 마이너스 (-) 기호가 깨지는 문제를 방지합니다.
# 예: -10같은 값이 폰트가 깨져서 마이너스 부호가 안보이는 문제를 방지합니다.

plt.rcParams["axes.unicode_minus"] = False
# 한글 글꼴 관련 경고가 너무 많이 출력되는것을 방지합니다.
warnings.filterwarnings("ignore", category=UserWarning)

In [40]:
# 같은 폴더에 jeju_bus.csv 가 있다고 가정합니다.
# 파일이 다른 경로에 있다면 아래 경로를 수정합니다. 예: pd.read_csv("data/jeju_bus.csv")

df = pd.read_csv("jeju_bus.csv")

# 원본 데이터는 보존하고, 모델링용으로는 복사본을 사용합니다.
# .copy() 를 쓰면 df_model을 바꿔도 원본 df는 영향을 받지 않습니다.

df_model = df.copy()

In [41]:
# 상위 5개 행을 확인합니다. 실제 값이 어떤 형태인지 눈으로 점검하는 단계입니다.
df.head()

,id,date,route_id,vh_id,route_nm,now_latitude,now_longitude,now_station,now_arrive_time,distance,next_station,next_latitude,next_longitude,next_arrive_time
0,0,2019-10-15,405136001,7997025,360-1,33.456267,126.551750,제주대학교입구,06시,266,제대마을,33.457724,126.554014,24
1,1,2019-10-15,405136001,7997025,360-1,33.457724,126.554014,제대마을,06시,333,제대아파트,33.458783,126.557353,36
2,2,2019-10-15,405136001,7997025,360-1,33.458783,126.557353,제대아파트,06시,415,제주대학교,33.459893,126.561624,40
3,3,2019-10-15,405136001,7997025,360-1,33.479705,126.543811,남국원(아라방면),06시,578,제주여자중고등학교(아라방면),33.484860,126.542928,42
4,4,2019-10-15,405136001,7997025,360-1,33.485662,126.494923,도호동,07시,374,은남동,33.485822,126.490897,64


In [42]:
# 데이터의 (행 개수, 열 개수) 를 확인합니다. 데이터 규모를 파악합니다.
df.shape

(210457, 14)

In [43]:
# 실제 컬럼명을 확인합니다. 이후 코드 (전처리, feature 목록 등)는 이 컬럼명을 기준으로 작성되어 있습니다.
# 만약 실제 컬럼명이 코드와 다르면, 이출력을 기준으로 코드의 컬럼명을 맞춰야 합니다.
df.columns

Index(['id', 'date', 'route_id', 'vh_id', 'route_nm', 'now_latitude',
       'now_longitude', 'now_station', 'now_arrive_time', 'distance',
       'next_station', 'next_latitude', 'next_longitude', 'next_arrive_time'],
      dtype='str')

In [44]:
# 컬럼별 결측치(비어있는 것) 개수를 확인합니다. 값이 클 수록 그 컬럼에 빈 값이 많다는 뜻입니다.
df.isnull().sum()


id                  0
date                0
route_id            0
vh_id               0
route_nm            0
now_latitude        0
now_longitude       0
now_station         0
now_arrive_time     0
distance            0
next_station        0
next_latitude       0
next_longitude      0
next_arrive_time    0
dtype: int64

In [45]:
# 각 컬럼의 자료형 (숫자/문자)과 결측치 여부를 한눈에 확인 합니다.
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 210457 entries, 0 to 210456
Data columns (total 14 columns):
 #   Column            Non-Null Count   Dtype  
---  ------            --------------   -----  
 0   id                210457 non-null  int64  
 1   date              210457 non-null  str    
 2   route_id          210457 non-null  int64  
 3   vh_id             210457 non-null  int64  
 4   route_nm          210457 non-null  str    
 5   now_latitude      210457 non-null  float64
 6   now_longitude     210457 non-null  float64
 7   now_station       210457 non-null  str    
 8   now_arrive_time   210457 non-null  str    
 9   distance          210457 non-null  int64  
 10  next_station      210457 non-null  str    
 11  next_latitude     210457 non-null  float64
 12  next_longitude    210457 non-null  float64
 13  next_arrive_time  210457 non-null  int64  
dtypes: float64(4), int64(5), str(5)
memory usage: 22.5 MB


In [46]:
# 수치형 컬럼의 요약 통계(평균, 표준편차, 최솟값/최댓값, 분위수)를 확인합니다.
df.describe()

,id,route_id,vh_id,now_latitude,now_longitude,distance,next_latitude,next_longitude,next_arrive_time
count,210457.000000,2.104570e+05,2.104570e+05,210457.000000,210457.000000,210457.000000,210457.000000,210457.000000,210457.000000
mean,105228.000000,4.052491e+08,7.988694e+06,33.434528,126.603451,490.256100,33.434711,126.603687,85.380824
std,60753.847139,9.132404e+04,6.774077e+03,0.102350,0.123961,520.563932,0.102224,0.123838,85.051170
min,0.000000,4.051360e+08,7.983000e+06,33.244382,126.473300,97.000000,33.244382,126.473300,6.000000
25%,52614.000000,4.051365e+08,7.983093e+06,33.325283,126.523900,291.000000,33.325283,126.524550,44.000000
50%,105228.000000,4.053201e+08,7.983431e+06,33.484667,126.551050,384.000000,33.484860,126.551050,66.000000
75%,157842.000000,4.053201e+08,7.997041e+06,33.500197,126.650322,542.000000,33.500228,126.650322,102.000000
max,210456.000000,4.053281e+08,7.997124e+06,33.556167,126.935188,7461.000000,33.556167,126.935188,2996.000000


In [47]:
df.columns

Index(['id', 'date', 'route_id', 'vh_id', 'route_nm', 'now_latitude',
       'now_longitude', 'now_station', 'now_arrive_time', 'distance',
       'next_station', 'next_latitude', 'next_longitude', 'next_arrive_time'],
      dtype='str')

In [48]:
df.columns[1:-1]

Index(['date', 'route_id', 'vh_id', 'route_nm', 'now_latitude',
       'now_longitude', 'now_station', 'now_arrive_time', 'distance',
       'next_station', 'next_latitude', 'next_longitude'],
      dtype='str')

In [49]:
target_col = df.columns[-1]
required_cols = df.columns[1:-1]
required_cols

Index(['date', 'route_id', 'vh_id', 'route_nm', 'now_latitude',
       'now_longitude', 'now_station', 'now_arrive_time', 'distance',
       'next_station', 'next_latitude', 'next_longitude'],
      dtype='str')

In [50]:
# 예측 대상(정답) 컬럼명을 변수로 지정해 둡니다. 이후 코드에서 이 변수를 일관되게 사용합니다.
target_col = "next_arrive_time"

# 전처리 입력에 반드시 있어야 하는 필수 컬럼 목록입니다.
# 이 목록은 아래 prepare_features() 안에서 "누락된 컬럼이 없는지" 검사하는 데 사용됩니다.
# 실제 데이터 컬럼명이 다르면 df.columns 출력을 확인한 뒤 이 목록을 수정합니다.
required_cols = [
    "route_id",
    "vh_id",
    "route_nm",
    "now_latitude",
    "now_longitude",
    "now_station",
    "now_arrive_time",
    "distance",
    "next_station",
    "next_latitude",
    "next_longitude",
    "date"
]

def prepare_features 함수
    
    학습 데이터와 예측 데이터에 동일한 전처리를 적용하는 함수 입니다.
    학습할 때와 예측할 때 같은 방식으로 데이터를 준비해야 모델이 일관된 
    형태의 입력을 받을 수 있으므로, 두 경우 모두 이 함수를 거치도록 합니다.

    주요 처리내용
    --------------------
    - date 컬럼에서 year, month, day, dayofweek 파생 컬럼을 생성합니다.
    - now_arrive_time 문자열("06시")에서 숫자 시간만 추출해 now_hour를 생성합니다.
    - 모델 입력에 사용하지 않는 id, date, now_arrive_time, target 컬럼을 재거합니다.

    Parameters
    --------------------
    df_input : pandas.DataFrame
        전처리 할 원본 데이터 입니다.

    Returns
    --------------------
    pandas.DataFrame
        모델 학습 또는 예측에 사용할 수 있도록 정리된 feature 데이터 입니다.

In [51]:
def prepare_features(df_input):
    
    # 원본을 직접 수정하지 않도록 복사본을 만들어 작업합니다.
    # (함수 밖의 데이터가 의도치 않게 바뀌는 것을 막기 위함입니다.)
    data = df_input.copy()
    
    # 필수 컬럼이 빠지지 않았는지 먼저 검사합니다.
    # 누락된 컬럼이 있으면 어떤 컬럼이 없는지 알려 주는 오류를 발생시켜 원인을 빨리 찾게 합니다.
    missing_cols = [col for col in required_cols if col not in data.columns]
    if missing_cols:
        raise ValueError(f"필수 컬럼이 누락되었습니다: {missing_cols}")
        
    # date를 datetime(날짜형)으로 변환한 뒤, 모델이 활용하기 좋은 숫자 파생 컬럼을 만듭니다.
    data["date"] = pd.to_datetime(data["date"])
    data["year"] = data["date"].dt.year                # 연도
    data["month"] = data["date"].dt.month              # 월
    data["day"] = data["date"].dt.day                  # 일
    data["dayofweek"] = data["date"].dt.dayofweek      # 요일 (월=0 ... 일=6)
    
    # now_arrive_time의 "06시" 같은 문자열에서 숫자 부분만 추출해 now_hour(정수 시간)로 만듭니다.
    # str.extract(r"(\d+)")는 문자열에서 연속된 숫자를 뽑아내는 처리입니다. ("06시" -> "06")
    data["now_hour"] = (
        data["now_arrive_time"].astype(str).str.extract(r"(\d+)").astype(float)
    )
    
    # 변환을 마친 원본 date, now_arrive_time 컬럼은 더 이상 필요 없으므로 feature에서 제외합니다.
    data = data.drop(columns=["date", "now_arrive_time"])
    
    # 정답(target) 컬럼이 들어 있으면 제거합니다. (입력 feature에는 정답이 포함되면 안 됩니다.)
    # 예측용 데이터에는 보통 target이 없으므로, 있을 때만 제거하도록 조건을 둡니다.
    if target_col in data.columns:
        data = data.drop(columns=[target_col])
        
    # 단순 식별 번호인 id 컬럼이 있으면 제거합니다. (예측에 도움이 되지 않는 정보입니다.)
    if "id" in data.columns:
        data = data.drop(columns=["id"])
        
    return data  # 정리된 feature DataFrame을 돌려줍니다.

In [52]:
# 1. 수동으로 글자를 타이핑하지 않고, 실제 데이터(df)의 컬럼 목록에서 자동으로 안전하게 가져옵니다.
required_cols = list(df.columns[1:-1])

# 2. 모델링용 복사본을 안전하게 생성합니다.
df_model = df.copy()

# 3. 이제 명칭이 100% 일치하므로 에러 없이 전처리 함수가 정상 작동합니다.
X_preprocess = prepare_features(df_model)

In [53]:
X_preprocess

,route_id,vh_id,route_nm,now_latitude,now_longitude,now_station,distance,next_station,next_latitude,next_longitude,year,month,day,dayofweek,now_hour
0,405136001,7997025,360-1,33.456267,126.551750,제주대학교입구,266,제대마을,33.457724,126.554014,2019,10,15,1,6.0
1,405136001,7997025,360-1,33.457724,126.554014,제대마을,333,제대아파트,33.458783,126.557353,2019,10,15,1,6.0
2,405136001,7997025,360-1,33.458783,126.557353,제대아파트,415,제주대학교,33.459893,126.561624,2019,10,15,1,6.0
3,405136001,7997025,360-1,33.479705,126.543811,남국원(아라방면),578,제주여자중고등학교(아라방면),33.484860,126.542928,2019,10,15,1,6.0
4,405136001,7997025,360-1,33.485662,126.494923,도호동,374,은남동,33.485822,126.490897,2019,10,15,1,7.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
210452,405328102,7983486,281-2,33.255783,126.577450,비석거리,528,삼아아파트,33.251896,126.574417,2019,10,28,0,21.0
210453,405328102,7983486,281-2,33.248595,126.568527,동문로터리,280,매일올레시장 7번입구,33.249753,126.565959,2019,10,28,0,21.0
210454,405328102,7983486,281-2,33.251891,126.560303,서귀포시 구 버스터미널,114,아랑조을거리 입구,33.251084,126.559551,2019,10,28,0,21.0
210455,405328102,7983486,281-2,33.251084,126.559551,아랑조을거리 입구,223,평생학습관,33.249504,126.558068,2019,10,28,0,21.0


In [54]:
# 전처리 후 모델에 들어갈 범주형 컬럼 목록입니다. (이름/종류를 나타내는 문자형 컬럼들)
categorical_features = [
    "route_nm",   # 버스노선 이름
    "now_station", # 현재 정류장
    "next_station" # 다음 정류장
]

# 범주형 전처리: 결측치를 최빈값(most_frequent)으로 채운 뒤 OneHotEncoder로 숫자화합니다.
# handle_unknown="ignore": 예측 시 학습 때 보지 못한 새 값이 와도 오류 없이 처리합니다.
# 버전 호환성을 위해 sparse_output / sparse 인자는 지정하지 않고 handle_unknown 만 사용합니다.
categorical_transformer = Pipeline(steps=[
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

# ColumnTransformer: 수치형 컬럼과 범주형 컬럼에 위에서 정의한 서로 다른 전처리를 한 번에 적용합니다.
# ("cat", 범주형 전처리, 범주형 컬럼들) 형태로 묶습니다.
preprocessor = ColumnTransformer(transformers=[
    ("cat", categorical_transformer, categorical_features)
])

In [55]:
# fit_transform()
# - fit: X 안의 범주형 값 목록을 학습합니다.
#   예: route_nm에는 어떤 노선명이 있는지, now_station에는 어떤 정류장이 있는지 확인
#
# - transform: 학습한 범주형 값 목록을 기준으로 OneHotEncoder 숫자 배열로 변환합니다.
X_cat_encoded = preprocessor.fit_transform(df_model)
X_cat_encoded

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 631371 stored elements and shape (210457, 720)>

In [56]:
# preprocessor는 ColumnTransformer 객체입니다.
# 위에서 preprocessor.fit_transform(X)를 실행하면,
# preprocessor는 X 안에 있는 범주형 컬럼들의 실제 값들을 학습합니다.
#
# 예를 들어 route_nm 컬럼에 다음 값들이 있었다고 가정합니다.
# - 201번
# - 202번
# - 300번
#
# OneHotEncoder는 이 값들을 각각 별도의 컬럼으로 바꿉니다.
# 예:
# - cat__route_nm_201번
# - cat__route_nm_202번
# - cat__route_nm_300번
#
# get_feature_names_out()은 이렇게 새로 만들어진 컬럼 이름 목록을 가져오는 함수입니다.
encoded_feature_names = preprocessor.get_feature_names_out()
encoded_feature_names

array(['cat__route_nm_201-11', 'cat__route_nm_201-12',
       'cat__route_nm_201-13', 'cat__route_nm_201-14',
       'cat__route_nm_201-15', 'cat__route_nm_201-16',
       'cat__route_nm_201-17', 'cat__route_nm_201-18',
       'cat__route_nm_201-21', 'cat__route_nm_201-22',
       'cat__route_nm_201-24', 'cat__route_nm_201-26',
       'cat__route_nm_201-27', 'cat__route_nm_281-1',
       'cat__route_nm_281-2', 'cat__route_nm_360-1',
       'cat__route_nm_360-12', 'cat__route_nm_360-2',
       'cat__route_nm_360-7', 'cat__route_nm_365-21',
       'cat__route_nm_365-22', 'cat__now_station_911의원',
       'cat__now_station_LH아파트', 'cat__now_station_가마초등학교',
       'cat__now_station_가흥동', 'cat__now_station_거로 입구',
       'cat__now_station_견월교', 'cat__now_station_계룡동',
       'cat__now_station_고도농원', 'cat__now_station_고래왓',
       'cat__now_station_고망난돌입구', 'cat__now_station_고산동산(광양방면)',
       'cat__now_station_고산동산(아라방면)', 'cat__now_station_고성리 구 성산농협',
       'cat__now_station_고성리 성산농협', 

In [57]:
# X_cat_encoded는 preprocessor.fit_transform(X)의 결과입니다.
# 즉, route_nm, now_station, next_station 같은 문자형 컬럼들이
# OneHotEncoder를 거쳐 0과 1로 변환된 결과입니다.
#
# 다만 OneHotEncoder의 결과는 일반적인 표(DataFrame)가 아니라
# sparse matrix 형태일 수 있습니다.
#
# sparse matrix는 대부분의 값이 0일 때 메모리를 아끼기 위해 사용하는 저장 방식입니다.
# 예를 들어 정류장 종류가 1,000개라면,
# 한 행에서 실제로 1이 되는 컬럼은 일부이고 대부분은 0입니다.
#
# 이런 데이터를 모두 표 형태로 저장하면 메모리를 많이 사용하므로,
# scikit-learn은 기본적으로 sparse matrix 형태로 저장할 수 있습니다.
#
# 하지만 수업 중에는 사람이 직접 눈으로 확인해야 하므로,
# toarray()를 사용해서 일반 numpy 배열 형태로 바꾼 뒤
# pandas DataFrame으로 변환합니다.
#
# columns=encoded_feature_names를 지정하면,
# 0과 1로 바뀐 각 컬럼에 사람이 이해할 수 있는 이름이 붙습니다.
X_cat_encoded_df = pd.DataFrame(
    X_cat_encoded.toarray(),         # sparse matrix를 일반 배열로 변환
    columns=encoded_feature_names   # OneHotEncoder가 만든 컬럼 이름을 DataFrame 컬럼명으로 사용
)

In [58]:
X_cat_encoded_df

,cat__route_nm_201-11,cat__route_nm_201-12,cat__route_nm_201-13,cat__route_nm_201-14,cat__route_nm_201-15,cat__route_nm_201-16,cat__route_nm_201-17,cat__route_nm_201-18,cat__route_nm_201-21,cat__route_nm_201-22,...,cat__next_station_화북주공아파트,cat__next_station_화북초등학교,cat__next_station_화성농장,cat__next_station_효돈농협하나로마트,cat__next_station_효돈중학교,cat__next_station_효돈초등학교,cat__next_station_효돈축구공원,cat__next_station_효례교,cat__next_station_흙통,cat__next_station_희진주유소
0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
210452,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
210453,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
210454,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
210455,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [59]:
# 정답값(target)은 prepare_features에서 제거되므로, 원본 df_model에서 따로 가져옵니다.
y = df_model[target_col]

y

0         24
1         36
2         40
3         42
4         64
          ..
210452    96
210453    50
210454    16
210455    38
210456    24
Name: next_arrive_time, Length: 210457, dtype: int64

In [60]:
X = prepare_features(df_model)

X

,route_id,vh_id,route_nm,now_latitude,now_longitude,now_station,distance,next_station,next_latitude,next_longitude,year,month,day,dayofweek,now_hour
0,405136001,7997025,360-1,33.456267,126.551750,제주대학교입구,266,제대마을,33.457724,126.554014,2019,10,15,1,6.0
1,405136001,7997025,360-1,33.457724,126.554014,제대마을,333,제대아파트,33.458783,126.557353,2019,10,15,1,6.0
2,405136001,7997025,360-1,33.458783,126.557353,제대아파트,415,제주대학교,33.459893,126.561624,2019,10,15,1,6.0
3,405136001,7997025,360-1,33.479705,126.543811,남국원(아라방면),578,제주여자중고등학교(아라방면),33.484860,126.542928,2019,10,15,1,6.0
4,405136001,7997025,360-1,33.485662,126.494923,도호동,374,은남동,33.485822,126.490897,2019,10,15,1,7.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
210452,405328102,7983486,281-2,33.255783,126.577450,비석거리,528,삼아아파트,33.251896,126.574417,2019,10,28,0,21.0
210453,405328102,7983486,281-2,33.248595,126.568527,동문로터리,280,매일올레시장 7번입구,33.249753,126.565959,2019,10,28,0,21.0
210454,405328102,7983486,281-2,33.251891,126.560303,서귀포시 구 버스터미널,114,아랑조을거리 입구,33.251084,126.559551,2019,10,28,0,21.0
210455,405328102,7983486,281-2,33.251084,126.559551,아랑조을거리 입구,223,평생학습관,33.249504,126.558068,2019,10,28,0,21.0


In [61]:
# train_test_split() 함수를 사용해 학습 데이터셋(Train set)과 검증용 데이터셋(Test set)을 분리합니다.
#
# - test_size=0.2: 전체 데이터의 20%를 검증용(Test)으로 사용하고, 나머지 80%를 학습용(Train)으로 사용합니다.
# - random_state=42: 무작위 분리 시 사용할 기준점(시드 값)입니다. 숫자가 같으면 언제 실행해도 똑같이 분리됩니다.
# - shuffle=True: 데이터를 나누기 전에 골고루 섞습니다. 시계열 데이터가 아니므로 섞어서 분리하는 것이 좋습니다.
X_train, X_test, y_train, y_test = train_test_split(X_preprocess, y, test_size=0.2, random_state=42)

#나뉘 학습/검증 데이터의 크기를 확인 합니다.
X_train.shape, X_test.shape

((168365, 15), (42092, 15))

In [62]:
# XGBoost 라이브러리에서 회귀 모델인 XGBRegressor를 가져옵니다.
# (버스 도착 시간은 '몇 초' 형태의 연속된 숫자를 예측하는 것이므로 회귀 알고리즘을 사용합니다.)
xgb_model = XGBRegressor()

model_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", xgb_model)
])
# fit(X, y) 함수를 호출하여 모델을 학습시킵니다.
# 학습용 feature(X_train)와 정답(y_train)을 입력받아 규칙과 패턴을 스스로 찾아냅니다.
model_pipeline = model_pipeline.fit(X_train, y_train)
print("모델이 학습이 완료되었습니다.")

모델이 학습이 완료되었습니다.


In [63]:
# 검증 데이터 (X_test)에 대한 예측값을 구합니다.  (전처리도 파이프라인이 자동으로 적용)
y_pred = model_pipeline.predict(X_test)

# 평가 지표를 계산합니다. 실제값(y_test.)와 예측값(y_pred)을 비교합니다.
mae = mean_absolute_error(y_test, y_pred) # 평균적으로 얼마나 틀리는지

# 지표를 소수점 4자리로 보기 좋게 출력합니다.
print(f'MAE: {mae:.4f}')

MAE: 26.2774


In [64]:
# 예측할 새로운 데이터
new_data = pd.DataFrame([
    {
        "date": "2019-10-21",
        "route_id": 405136001,
        "vh_id": 7997025,
        "route_nm": "360-1",
        "now_latitude": 33.456267,
        "now_longitude": 126.551750,
        "now_station": "제주대학교입구",
        "now_arrive_time": "08시",
        "distance": 300.0,
        "next_station": "제대마을",
        "next_latitude": 33.457724,
        "next_longitude": 126.554014
    }
])

In [65]:
# 학습때와 동일한 prepare_features 함수를 예측용 데이터에도 적용합니다.
# 같은 전처리를 거쳐야 학습때와 feature 구조가 일치해 모델이 올바르게 예측합니다.
new_data_prepared = prepare_features(new_data)

# 학습된 파이프라인으로 예측합니다. (내부 전처리도 함깨 자동적용)
new_pred = model_pipeline.predict(new_data_prepared)

# 예측 결과를 보기 좋게 원본 예시 DataFrame 에 새 컬럼으로 붙입니다.
result_df = new_data.copy()
result_df['predicted_next_arrive_time'] = new_pred  # 모델이 예상한 다음 정류장 도착 시간
result_df['predicted_next_arrive_time'] = result_df['predicted_next_arrive_time'].round(2) # 소수점 2자리로 정리

# 핵심 컬럼만 골라 예측 결과를 확인 합니다.
result_df[[
    "route_nm",
    "now_station",
    "next_station",
    "now_arrive_time",
    "distance",
    "predicted_next_arrive_time"    
]]

,route_nm,now_station,next_station,now_arrive_time,distance,predicted_next_arrive_time
0,360-1,제주대학교입구,제대마을,08시,300.0,37.950001
